In [ ]:
# =========================================================
# Dhaka Traffic Signal: Side-by-Side Animation (Team 16)
# Same traffic, two signal plans: baseline vs optimized.
# Inputs come from our own simulation model (arrival rates,
# vehicle mix, headways, signal plans).
# =========================================================
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle, FancyBboxPatch, Circle

try:
    from IPython.display import HTML, display
except ImportError:                      # running as a normal script
    HTML = display = None

matplotlib.rcParams['animation.embed_limit'] = 200

# ================= SETTINGS (edit these) =================
SEED = 3                 # same seed = same traffic in both panels
DURATION = 360           # simulated seconds shown (1 frame = 1 second)
HOLD = 40                # extra frames at the end to show the result

# vehicles per minute on each approach (from the published counts)
ARRIVAL_PER_MIN = {"N": 667 / 60, "S": 614 / 60, "E": 583 / 60, "W": 0.0}

PLAN_BASE = (42, 58)     # baseline: NS green, EW green (seconds)
PLAN_BEST = (30, 25)     # best plan found by our grid search
LOST = 6                 # seconds between phases (yellow + all-red)
YELLOW = 3               # first seconds of that gap show yellow

# Dhaka traffic mix. share = fraction of traffic, headway = seconds a vehicle
# needs to leave the stop line (same values as the SimPy model),
# speed / length / width are drawing units (1 unit is about 2 metres).
KINDS = {
    "Car":      dict(share=0.30, headway=2.0, speed=5.0, length=2.8, width=1.5,
                     colors=['#264653', '#2a9d8f', '#e9c46a', '#f4a261', '#6d597a', '#b5838d']),
    "Bus":      dict(share=0.10, headway=3.5, speed=4.0, length=5.6, width=1.9,
                     colors=['#e63946', '#f77f00']),
    "CNG":      dict(share=0.20, headway=2.0, speed=4.6, length=2.3, width=1.3,
                     colors=['#2d9d4f']),
    "Rickshaw": dict(share=0.20, headway=2.5, speed=2.6, length=2.1, width=1.1,
                     colors=['#ff6b9d', '#4cc9f0', '#ffd166', '#9d4edd']),
    "Bike":     dict(share=0.20, headway=1.2, speed=5.6, length=1.6, width=0.7,
                     colors=['#212529', '#495057']),
}
# =========================================================

CENTER, ROAD_HALF = 50, 8
S_STOP = 42.0            # distance from the map edge to the stop line
GAP = 0.8                # space kept between vehicles
DT = 0.25                # simulation step (seconds)
SUB = int(round(1 / DT))
DIRS = list(ARRIVAL_PER_MIN)
GROUP = {"N": "NS", "S": "NS", "E": "EW", "W": "EW"}


# ---------- traffic generation (same list is used by both plans) ----------
def make_arrivals(seed):
    rng = np.random.default_rng(seed)
    names = list(KINDS)
    probs = [KINDS[k]['share'] for k in names]
    out = {}
    for d, per_min in ARRIVAL_PER_MIN.items():
        lst, t = [], 0.0
        if per_min > 0:
            while True:
                t += rng.exponential(60.0 / per_min)      # Poisson arrivals
                if t > DURATION:
                    break
                kind = names[rng.choice(len(names), p=probs)]
                color = str(rng.choice(KINDS[kind]['colors']))
                lst.append(dict(t=t, kind=kind, color=color))
        out[d] = lst
    return out


# ---------- signal ----------
def lights(t, plan):
    g_ns, g_ew = plan
    cycle = g_ns + g_ew + 2 * LOST
    u = t % cycle
    ns = ew = 'red'
    if u < g_ns:
        ns, text = 'green', f"NS green, {int(g_ns - u) + 1} s left"
    elif u < g_ns + LOST:
        ns = 'yellow' if u - g_ns < YELLOW else 'red'
        text = "NS yellow" if ns == 'yellow' else "all red"
    elif u < g_ns + LOST + g_ew:
        ew, text = 'green', f"EW green, {int(g_ns + LOST + g_ew - u) + 1} s left"
    else:
        ew = 'yellow' if u - (g_ns + LOST + g_ew) < YELLOW else 'red'
        text = "EW yellow" if ew == 'yellow' else "all red"
    return {'N': ns, 'S': ns, 'E': ew, 'W': ew}, text


# ---------- one intersection ----------
class Junction:
    def __init__(self, plan, arrivals):
        self.plan, self.arr, self.t = plan, arrivals, 0.0
        self.cars = {d: [] for d in DIRS}
        self.ptr = {d: 0 for d in DIRS}
        self.next_release = {d: 0.0 for d in DIRS}
        self.waits = []                                   # delays of vehicles already released

    def step(self, dt):
        t = self.t
        light, _ = lights(t, self.plan)
        for d in DIRS:
            lst, arr, i = self.cars[d], self.arr[d], self.ptr[d]
            # a vehicle enters when its time has come and there is room
            if i < len(arr) and arr[i]['t'] <= t:
                if not lst or lst[-1]['s'] - lst[-1]['len'] >= GAP:
                    sp = KINDS[arr[i]['kind']]
                    lst.append(dict(kind=arr[i]['kind'], s=0.0, len=sp['length'], w=sp['width'],
                                    vmax=sp['speed'], color=arr[i]['color'],
                                    free=arr[i]['t'] + S_STOP / sp['speed'], rel=False))
                    self.ptr[d] += 1
            # move vehicles, front to back
            for idx, v in enumerate(lst):
                lead = lst[idx - 1] if idx else None
                limit = lead['s'] - lead['len'] - GAP if lead else 1e9
                if not v['rel']:
                    limit = min(limit, S_STOP)            # wait at the stop line
                v['s'] = max(v['s'], min(v['s'] + v['vmax'] * dt, limit))
                at_line = v['s'] >= S_STOP - 1e-6
                if (not v['rel'] and at_line and light[d] == 'green'
                        and t >= self.next_release[d] and (idx == 0 or lead['rel'])):
                    v['rel'] = True                       # released: it may cross now
                    self.next_release[d] = t + KINDS[v['kind']]['headway']
                    self.waits.append(max(0.0, t - v['free']))
            lst[:] = [v for v in lst if v['s'] - v['len'] <= 102]
        self.t += dt

    def stats(self):
        """Average delay (s) of all vehicles so far, and how many are delayed right now."""
        t, current = self.t, []
        for d in DIRS:
            for v in self.cars[d]:
                if not v['rel'] and t - v['free'] > 0:
                    current.append(t - v['free'])
            for a in self.arr[d][self.ptr[d]:]:           # waiting outside, road is full
                if a['t'] > t:
                    break
                late = t - (a['t'] + S_STOP / KINDS[a['kind']]['speed'])
                if late > 0:
                    current.append(late)
        allw = self.waits + current
        return (float(np.mean(allw)) if allw else 0.0), len(current)


def run_plan(plan, arrivals):
    jn, history = Junction(plan, arrivals), []
    for _ in range(DURATION + 1):
        for _ in range(SUB):
            jn.step(DT)
        light, text = lights(jn.t, plan)
        avg, queue = jn.stats()
        cars = [(d, v['s'], v['len'], v['w'], v['color']) for d in DIRS for v in jn.cars[d]]
        history.append(dict(cars=cars, light=light, text=text, avg=avg, queue=queue))
    return history


# ---------- drawing ----------
def vehicle_box(d, s, length, width):
    sc = s - length / 2                                   # centre along the road
    if d == 'N':   cx, cy, vert = CENTER + 4, 100 - sc, True      # southbound, left-hand traffic
    elif d == 'S': cx, cy, vert = CENTER - 4, sc, True            # northbound
    elif d == 'E': cx, cy, vert = 100 - sc, CENTER - 4, False     # westbound
    else:          cx, cy, vert = sc, CENTER + 4, False           # eastbound
    if vert:
        return cx - width / 2, cy - length / 2, width, length
    return cx - length / 2, cy - width / 2, length, width


def draw_trees(ax):
    rng = np.random.default_rng(3)
    trees = []
    while len(trees) < 26:
        x, y = rng.uniform(3, 97, 2)
        if abs(x - CENTER) < 13 or abs(y - CENTER) < 13:
            continue                                      # keep roads and signals clear
        if any((x - a) ** 2 + (y - b) ** 2 < 49 for a, b in trees):
            continue
        trees.append((x, y))
    greens = [('#1f5a2a', '#2f7d3a'), ('#245f31', '#3a9246'), ('#1b4d28', '#2c7536')]
    for n, (x, y) in enumerate(trees):
        dark, light = greens[n % 3]
        ax.add_patch(Rectangle((x - 0.4, y - 2.6), 0.8, 2.2, color='#6b4f2a', zorder=3))
        ax.add_patch(Circle((x, y), 2.6, color=dark, zorder=4))
        ax.add_patch(Circle((x - 0.9, y + 0.9), 1.7, color=light, zorder=5))
        ax.add_patch(Circle((x + 1.0, y + 0.4), 1.3, color=light, alpha=0.8, zorder=5))


def draw_scene(ax):
    ax.set_facecolor('#3f7f3a')
    rng = np.random.default_rng(1)
    gx, gy = rng.uniform(0, 100, 320), rng.uniform(0, 100, 320)
    keep = (np.abs(gx - CENTER) > ROAD_HALF) & (np.abs(gy - CENTER) > ROAD_HALF)
    ax.scatter(gx[keep], gy[keep], s=3, color='#4f9447', alpha=0.6, zorder=1)
    draw_trees(ax)
    road = '#44474d'
    ax.add_patch(Rectangle((0, CENTER - ROAD_HALF), 100, 2 * ROAD_HALF, color=road, zorder=2))
    ax.add_patch(Rectangle((CENTER - ROAD_HALF, 0), 2 * ROAD_HALF, 100, color=road, zorder=2))
    ax.add_patch(Rectangle((CENTER - ROAD_HALF, CENTER - ROAD_HALF), 2 * ROAD_HALF, 2 * ROAD_HALF,
                           color='#4d5157', zorder=3))
    for xs, ys in (([0, 42], [50, 50]), ([58, 100], [50, 50]), ([50, 50], [0, 42]), ([50, 50], [58, 100])):
        ax.plot(xs, ys, '--', color='#f4f1de', lw=1, zorder=4)
    for i in range(6):                                    # zebra crossings
        o = 42.8 + 2.4 * i
        ax.add_patch(Rectangle((o, 59), 1.2, 3, color='#e9ecef', zorder=4))
        ax.add_patch(Rectangle((o, 38), 1.2, 3, color='#e9ecef', zorder=4))
        ax.add_patch(Rectangle((59, o), 3, 1.2, color='#e9ecef', zorder=4))
        ax.add_patch(Rectangle((38, o), 3, 1.2, color='#e9ecef', zorder=4))
    for xs, ys in (([50, 58], [58, 58]), ([42, 50], [42, 42]), ([58, 58], [42, 50]), ([42, 42], [50, 58])):
        ax.plot(xs, ys, color='white', lw=2.2, zorder=4)  # stop lines
    ax.set_xlim(0, 100); ax.set_ylim(0, 100)
    ax.set_aspect('equal'); ax.axis('off')


def make_signal_heads(ax):
    spots = {'N': (59.5, 60), 'S': (37.9, 34), 'E': (59.5, 34), 'W': (37.9, 60)}
    lamps = {}
    for d, (sx, sy) in spots.items():
        ax.add_patch(Rectangle((sx, sy), 2.6, 6.2, color='#161616', zorder=6))
        r = Circle((sx + 1.3, sy + 5.0), 0.85, zorder=7)
        y = Circle((sx + 1.3, sy + 3.1), 0.85, zorder=7)
        g = Circle((sx + 1.3, sy + 1.2), 0.85, zorder=7)
        for c in (r, y, g):
            ax.add_patch(c)
        lamps[d] = (r, y, g)
    return lamps


def set_lamps(lamps, light):
    for d, (r, y, g) in lamps.items():
        s = light[d]
        r.set_facecolor('#ff3b30' if s == 'red' else '#4a1512')
        y.set_facecolor('#ffcc00' if s == 'yellow' else '#4a4010')
        g.set_facecolor('#34e07a' if s == 'green' else '#0f3a1f')


# ---------- build everything ----------
arrivals = make_arrivals(SEED)
hist = [run_plan(PLAN_BASE, arrivals), run_plan(PLAN_BEST, arrivals)]
titles = [f"Baseline plan: NS {PLAN_BASE[0]} s / EW {PLAN_BASE[1]} s",
          f"Optimized plan: NS {PLAN_BEST[0]} s / EW {PLAN_BEST[1]} s"]
colors_line = ['#ff8a65', '#4dd0b1']

fig = plt.figure(figsize=(12, 7.8), dpi=72)
fig.patch.set_facecolor('#16212b')
gs = GridSpec(2, 2, height_ratios=[3.3, 1.05], hspace=0.16, wspace=0.04,
              left=0.06, right=0.97, top=0.9, bottom=0.09)
panels = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])]
axc = fig.add_subplot(gs[1, :])

lamp_sets, phase_txt, stat_txt = [], [], []
for ax, title in zip(panels, titles):
    draw_scene(ax)
    ax.set_title(title, color='#e8eef2', fontsize=12, pad=6)
    lamp_sets.append(make_signal_heads(ax))
    phase_txt.append(ax.text(1.5, 98.5, '', color='white', fontsize=9, va='top', zorder=12,
                             bbox=dict(boxstyle='round,pad=0.3', fc='#000000', alpha=0.55, ec='none')))
    stat_txt.append(ax.text(1.5, 1.5, '', color='white', fontsize=9, va='bottom', zorder=12,
                            bbox=dict(boxstyle='round,pad=0.3', fc='#000000', alpha=0.55, ec='none')))

fig.suptitle("Same traffic, two signal plans: which one keeps Dhaka moving?",
             color='white', fontsize=15, fontweight='bold', y=0.975)

axc.set_facecolor('#1d2b36')
axc.set_xlim(0, DURATION)
axc.set_ylim(0, max(max(h['queue'] for h in hh) for hh in hist) + 3)
axc.set_xlabel("Time (s)", color='#cfd8dc', fontsize=9)
axc.set_ylabel("Vehicles delayed", color='#cfd8dc', fontsize=9)
axc.tick_params(colors='#cfd8dc', labelsize=8)
for sp in axc.spines.values():
    sp.set_color('#455a64')
axc.grid(color='#37474f', lw=0.5, alpha=0.6)
lines = [axc.plot([], [], color=c, lw=2, label=l)[0]
         for c, l in zip(colors_line, ("Baseline", "Optimized"))]
dots = [axc.plot([], [], 'o', color=c, ms=6)[0] for c in colors_line]
axc.legend(loc='upper left', fontsize=8, facecolor='#1d2b36', edgecolor='#455a64', labelcolor='white')
axc.set_title("Live count of vehicles delayed at the signal", color='#e8eef2', fontsize=10, pad=4)

fig.text(0.5, 0.02,
         "Bus = long and slow to clear, CNG = green, rickshaw = slow, bike = small and quick. "
         "Left-hand traffic as in Bangladesh. Delay is measured against free-flow travel.",
         color='#9fb3c0', fontsize=8.5, ha='center')

dynamic = []


def animate(frame):
    k = min(frame, DURATION)
    for a in dynamic:
        a.remove()
    dynamic.clear()
    for i, ax in enumerate(panels):
        snap = hist[i][k]
        for d, s, length, width, color in snap['cars']:
            x, y, w, h = vehicle_box(d, s, length, width)
            p = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0,rounding_size=0.35',
                               fc=color, ec='#0b0b0b', lw=0.8, zorder=8)
            ax.add_patch(p)
            dynamic.append(p)
        set_lamps(lamp_sets[i], snap['light'])
        phase_txt[i].set_text(f"t = {k} s   |   {snap['text']}")
        stat_txt[i].set_text(f"avg delay {snap['avg']:.1f} s   |   {snap['queue']} vehicles delayed")
        xs = list(range(k + 1))
        ys = [hist[i][j]['queue'] for j in xs]
        lines[i].set_data(xs, ys)
        dots[i].set_data([k], [ys[-1]])
    if frame >= DURATION:
        wb, wo = hist[0][DURATION]['avg'], hist[1][DURATION]['avg']
        cut = 100 * (wb - wo) / wb if wb > 0 else 0
        axc.set_title(f"Result after {DURATION} s: optimized plan cuts the average delay by {cut:.0f}% "
                      f"({wb:.1f} s to {wo:.1f} s)", color='#4dd0b1', fontsize=11, fontweight='bold', pad=4)


anim = FuncAnimation(fig, animate, frames=DURATION + HOLD + 1, interval=55, blit=False)

if display is not None:
    try:
        display(HTML(anim.to_html5_video()))      # small file, plays like a video
    except Exception:
        display(HTML(anim.to_jshtml()))           # fallback if ffmpeg is missing
    plt.close(fig)
else:
    plt.show()

# Optional: save a file for your slides or GitHub
# anim.save('traffic_animation.gif', writer='pillow', fps=18)